## Ocean Cubes Visualization: Mediterranean Biophysics (Remote S3 GeoZarr)


In [ ]:
# ==============================================================================
# 0. Start CORS & Local Zarr Proxy (works locally and on EOXHub / JupyterHub)
# ==============================================================================
# - Remote CloudFerro S3 GeoZarrs need CORS headers on HEAD requests.
# - Local cluster Zarr v2/v3 stores (e.g. `bucket/mediterranean_fronts_v3.zarr`)
#   cannot be served via Jupyter's `/files/` route because `ContentsManager`
#   blocks hidden dot-files (`.zmetadata`, `.zgroup`, `.zarray`, `.zattrs`).
# - `cors_proxy.py` handles both: upstream S3 paths (`/...`) and cluster-local
#   files (`/local/...`, with dot-files + HTTP Range/HEAD support).
# - On EOXHub, browser requests reach `127.0.0.1:8010` inside the pod through
#   `jupyter-server-proxy` at `<HUB_ORIGIN><JUPYTERHUB_SERVICE_PREFIX>proxy/8010`.

import http.client
import json
import os
import socket
import subprocess
import sys
import time

PROXY_PORT = 8010
HUB_ORIGIN = os.environ.get("HUB_ORIGIN", "https://jupyterhub.hub-otc-sc.eox.at").rstrip("/")
service_prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX")

# Serve `/local/...` from the Jupyter Server root / $HOME (matching Jupyter's `/files/` root)
# so notebooks inside subfolders (e.g. `~/ocean_hackathon/4_Visualizations`) can still read `~/bucket/...`.
_home = os.path.abspath(os.path.expanduser("~"))
_cwd = os.path.abspath(os.getcwd())
LOCAL_ROOT = os.path.abspath(
    os.environ.get("JUPYTER_SERVER_ROOT")
    or (_home if os.path.commonpath([_home, _cwd]) == _home else _cwd)
)

if service_prefix:
    PROXY_BASE = f"{HUB_ORIGIN}/{service_prefix.strip('/')}/proxy/{PROXY_PORT}"
else:
    PROXY_BASE = f"http://localhost:{PROXY_PORT}"


def _check_proxy_health(port: int):
    try:
        conn = http.client.HTTPConnection("127.0.0.1", port, timeout=3)
        conn.request("GET", "/healthz")
        resp = conn.getresponse()
        body = resp.read().decode("utf-8", errors="replace")
        conn.close()
        if resp.status == 200:
            data = json.loads(body)
            if "local_root" in data:
                return data
    except Exception:
        pass
    return None


with socket.socket() as s:
    port_in_use = s.connect_ex(("127.0.0.1", PROXY_PORT)) == 0

health = _check_proxy_health(PROXY_PORT) if port_in_use else None

# Restart cors_proxy.py if an older instance lacks `/local/` or was started with a different `--local-root`
if port_in_use and (
    health is None or os.path.abspath(health.get("local_root", "")) != LOCAL_ROOT
):
    if "proxy" in globals() and proxy.poll() is None:
        proxy.terminate()
        proxy.wait(timeout=5)
    else:
        subprocess.run(
            ["pkill", "-f", f"cors_proxy.py.*--port {PROXY_PORT}"],
            check=False,
        )
    for _ in range(20):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", PROXY_PORT)) != 0:
                port_in_use = False
                health = None
                break
        time.sleep(0.1)

if not port_in_use:
    proxy = subprocess.Popen(
        [
            sys.executable,
            "cors_proxy.py",
            "--port",
            str(PROXY_PORT),
            "--local-root",
            LOCAL_ROOT,
        ]
    )
    for _ in range(30):
        health = _check_proxy_health(PROXY_PORT)
        if health is not None:
            break
        time.sleep(0.1)
    print(f"Started CORS & Local Zarr proxy on 127.0.0.1:{PROXY_PORT} (pid {proxy.pid})")
elif health is not None:
    print(f"Proxy already running on 127.0.0.1:{PROXY_PORT}")
else:
    print(
        f"WARNING: Port {PROXY_PORT} is occupied by another process. "
        "Stop that process or change PROXY_PORT above."
    )

# --- Diagnostic Checks ---
if health:
    print(f"  [OK] Internal health check : {health}")
try:
    conn = http.client.HTTPConnection("127.0.0.1", PROXY_PORT, timeout=5)
    conn.request("HEAD", "/EarthCODE/OSCAssets/med_cubes/ocean-med-biophysics.zarr/.zmetadata")
    up_resp = conn.getresponse()
    up_resp.read()
    conn.close()
    print(f"  [OK] Upstream S3 HEAD test : HTTP {up_resp.status} (CORS header: {up_resp.getheader('Access-Control-Allow-Origin')})")
except Exception as exc:
    print(f"  [WARN] Upstream S3 check failed: {exc}")

print(f"\nBrowser Proxy Base URL: {PROXY_BASE}")
if service_prefix:
    print(f"Test EOXHub browser routing by opening: {PROXY_BASE}/healthz")

In [ ]:
# ==============================================================================
# 1. Remote S3 GeoZarr: Mediterranean Biophysics Cube (Web Mercator)
# ==============================================================================

import importlib
import ipyeoxelements
import ipyeoxelements.generated

# Reset template to clean state if re-executing cell
if "eox-timecontrol-date" in ipyeoxelements.EOxTimecontrol._esm_template:
    importlib.reload(ipyeoxelements.generated)
    importlib.reload(ipyeoxelements)

from ipyeoxelements import EOxMap, EOxLayercontrol, EOxTimecontrol
from ipywidgets import HBox, Layout, VBox

data = f"{PROXY_BASE}/EarthCODE/OSCAssets/med_cubes/ocean-med-biophysics.zarr"
print("Remote GeoZarr URL:", data)

# ------------------------------------------------------------------------------
# Layer Configurations
# ------------------------------------------------------------------------------
layers = [
    {"type": "Tile", "source": {"type": "OSM"}},
    {
        "type": "WebGLTile",
        "properties": {
            "id": "sea-ice-cube",
            "title": "Mediterranean Biophysics GeoZarr",
            "layerControlExpand": True,
            "layerControlToolsExpand": True,
            "layerConfig": {
                "type": "style",
                "style": True,
                "autofill": True,
            },
        },
        "source": {
            "type": "GeoZarr",
            "crossOrigin": "anonymous",
            "url": data,
            "bands": [],
        },
        "style": {
            "variables": {
                "min": 0,
                "max": 40,
            },
            "color": [
                "interpolate",
                ["linear"],
                ["band", 1],
                ["var", "min"],
                ["color", 0, 210, 255, ["band", 2]],
                [
                    "+",
                    ["var", "min"],
                    ["*", ["-", ["var", "max"], ["var", "min"]], 0.3333333333333333],
                ],
                ["color", 254, 217, 118, ["band", 2]],
                [
                    "+",
                    ["var", "min"],
                    ["*", ["-", ["var", "max"], ["var", "min"]], 0.6666666666666666],
                ],
                ["color", 253, 141, 60, ["band", 2]],
                ["var", "max"],
                ["color", 189, 0, 38, ["band", 2]],
            ],
        },
    },
]

# Ensure EOxTimecontrol includes its child components as specified in the Storybook story:
time_control_file = "./timecontrol.js"
with open(time_control_file, "r") as f:
    time_control_element = f.read()

if "eox-timecontrol-date" not in EOxTimecontrol._esm_template:
    EOxTimecontrol._esm_template = EOxTimecontrol._esm_template.replace(
        "view.el.appendChild(el);", time_control_element
    )

# ------------------------------------------------------------------------------
# Create and Display Map & Widgets
# ------------------------------------------------------------------------------
eox_map = EOxMap(
    id="main-map",
    center=[15, 45],
    zoom=4,
    layout=Layout(width="100%", height="500px"),
)
eox_map.layers = layers

time_control = EOxTimecontrol(
    for_="#main-map",
    init_date=["first"],
    layout=Layout(width="100%"),
)

control = EOxLayercontrol(
    for_="#main-map",
    tools=["config", "opacity"],
    layout=Layout(flex="1", height="550px", width="300px", min_width="250px"),
)

map_container = VBox(
    [eox_map, time_control],
    layout=Layout(flex="2", display="flex", flex_direction="column", gap="10px"),
)

HBox(
    [control, map_container],
    layout=Layout(width="100%", display="flex", gap="16px", align_items="flex-start"),
)

In [ ]:
# Stop the local CORS & Zarr proxy when you are done (the maps above stop loading data).
if "proxy" in globals() and proxy.poll() is None:
    proxy.terminate()
    print("Stopped proxy.")